#### Importing  Dependencies

In [7]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.messages import SystemMessage, HumanMessage
from langchain.tools import tool

#### Tools

In [8]:
import requests
import trafilatura
from sympy import sympify
from enh_web_search import websearch

@tool('web_search', description='A tool to perform a web search and retrieve relevant information. Input should be a search query as a string, e.g. "What is the capital of France?".')
def web_search(query: str) -> str:
    return websearch(query)
   
@tool('calculator', description='A tool to evaluate mathematical expressions. Input should be a valid math expression as a string, e.g. "2 + 2 * (3 - 1)".')
def calculator(expression: str) -> str:
    try:
        result = sympify(expression)
        return str(result)
    except Exception as e:
        return f"Error: {str(e)}"

@tool('fetch_url', description='A tool to fetch and extract text content from a given URL. Input should be a valid URL as a string, e.g. "https://www.example.com".')
def fetch_url(url: str) -> str:
    try:
        downloaded = trafilatura.fetch_url(url)
        result = trafilatura.extract(downloaded, include_comments=False, include_tables=False)
        return result if result else "No content extracted."
    except Exception as e:
        return f"Error: {str(e)}"

@tool('code_execution', description='A tool to execute code snippets. Input should be a valid code as a string.')
def code_execution(code: str, language: str = "python", input: str = "")-> str:
    compilers = {
        "python": "python-3.14",
        "cpp": "cpp-g++15",
        "c": "c-gcc15",
        "java": "java-openjdk25",
        "javascript": "typescript-deno",
        "typescript": "typescript-deno",
        "go": "go-1.26",
        "rust": "rust-1.93",
    }

    response = requests.post(
        "https://api.onlinecompiler.io/api/run-code-sync/",
        headers={
            "Authorization": os.getenv("ONLINECOMPILER_API_KEY")
        },
        json={
            "compiler": compilers[language],
            "code": code,
            "input": input
        }
    )
    return response.json().get("output", "")
    
@tool('chart_visualization', description='A tool to create charts from data. Input should be a JSON string containing the data and chart type, e.g. {"type": "bar", "data": {"labels": ["A", "B"], "values": [10, 20]}}.')
def chart_visualization(data: str) -> str:
    pass


#### LLM Initialization

In [17]:
 # model = init_chat_model("gemma-4-31b-it", model_provider="google_genai", api_key=os.getenv("GEMINI_API_KEY"))
model = init_chat_model("mistral-small-latest", model_provider="mistralai", api_key=os.getenv("MISTRAL_API_KEY"))

#### Agent Initialization

In [18]:
agent = create_agent(model=model, tools=[web_search, calculator, fetch_url, code_execution, chart_visualization])

In [ ]:
query = "What is the best French painter?"
for chunk in agent.stream({"messages": [HumanMessage(content=query)]}):
    if isinstance(chunk, dict) and "messages" in chunk:
        last = chunk["messages"][-1]
        print(getattr(last, "content", last), end="", flush=True)
    else:
        print(getattr(chunk, "content", chunk), end="", flush=True)

InvalidUpdateError: Expected dict, got What is the best French painter?
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/INVALID_GRAPH_NODE_RETURN_VALUE

#### Prompt Enginnering

In [11]:
date = "2026-05-29"
SYSTEM_PROMPT = f"""You are an evidence-based research assistant.

Current date: {date}

When tool results are available:

- Treat tool results as the authoritative source.
- Use retrieved information even if it differs from your training knowledge.
- Never claim a source is speculative unless the source itself indicates speculation.
- Cite retrieved evidence when making claims.
- If multiple retrieved sources agree, assume the information is correct.
- Do not discard retrieved information because of unfamiliar version numbers, dates, products, or companies."""

In [12]:
message = """What is the capital of India?"""

In [13]:
# from enh_web_search import websearch

In [14]:
print(websearch("youtube video on claude opus 4.8"))

[1] https://aws.amazon.com/about-aws/whats-new/2026/05/claude-opus-4.8-aws/
Claude Opus 4.8 is now available on AWS AWS now offers Claude Opus 4.8 -- Anthropic's most capable generally available model to date -- delivering meaningful advances across agentic coding, professional knowledge work, and long-running autonomous tasks for developers and enterprises building production AI applications. Claude Opus 4.8 can perform longer autonomous runs, deeper reasoning, and consistency to be trusted with production work. For coding, the Opus 4.8 reads codebases like an engineer, plans before it edits, and holds context across long sessions in real repositories. For agentic tasks, it is better at finding paths around obstacles instead of stalling, recovering from its own errors, and knowing when to ask for help versus when to keep going. For knowledge work, it better synthesizes across long documents and complex sources, self-checks its output, and delivers structured deliverables that hold up 

In [15]:
messages = [
    SystemMessage(SYSTEM_PROMPT),
    HumanMessage(message)
]
response = agent.invoke({"messages": messages})
print(response)

{'messages': [SystemMessage(content='You are an evidence-based research assistant.\n\nCurrent date: 2026-05-29\n\nWhen tool results are available:\n\n- Treat tool results as the authoritative source.\n- Use retrieved information even if it differs from your training knowledge.\n- Never claim a source is speculative unless the source itself indicates speculation.\n- Cite retrieved evidence when making claims.\n- If multiple retrieved sources agree, assume the information is correct.\n- Do not discard retrieved information because of unfamiliar version numbers, dates, products, or companies.', additional_kwargs={}, response_metadata={}, id='739fae20-0995-41b1-835f-5f57425b6f09'), HumanMessage(content='What is the capital of India?', additional_kwargs={}, response_metadata={}, id='9443923d-192b-4e62-bd72-3bef0b3ff60d'), AIMessage(content='The capital of India is **New Delhi**.', additional_kwargs={}, response_metadata={'token_usage': {'prompt_tokens': 592, 'total_tokens': 602, 'completion

In [16]:
def extract_output(response):
    last = response["messages"][-1].content
    if isinstance(last, list):
        return " ".join(
            b.get("text", "") for b in last if b.get("type") == "text"
        ).strip()
    return last

print(extract_output(response))

The capital of India is **New Delhi**.
